# PyTorch Going Modular

Turn notebook code into Python scripts

Each section has its imports

## **0. Globals**

Imports, Constants

In [44]:
%%capture
!pip install torchinfo

In [45]:
import torch
import torchvision
from torchvision import transforms
from torchinfo import summary

In [46]:
!mkdir modules

mkdir: cannot create directory ‘modules’: File exists


In [47]:
DEVICE = "mps" if torch.mps.is_available() else "cpu"
DEVICE = "cuda" if torch.cuda.is_available() else DEVICE
DEVICE

'cuda'

In [48]:
RANDOM_SEED = 42

## **1. Create Datasets and DataLoaders (data_setup.py)**

In [49]:
%%writefile modules/data_setup.py
"""
Contains functionality for creating PyTorch DataLoaders for
image classification data from torchvision datasets.
"""
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    dataset_name: str,
    data_path: str,
    train_transform: transforms.Compose,
    test_transform: transforms.Compose,
    batch_size: int,
    num_workers: int = NUM_WORKERS
):
    """
    Creates training and testing DataLoaders.

    Takes a torchvision dataset class name (e.g., CIFAR10) and creates
    PyTorch Datasets and DataLoaders for training and testing.

    Args:
        dataset_name: A torchvision dataset name (e.g., CIFAR10).
        data_path: Path to store/load the dataset.
        train_transform: torchvision transforms for training data.
        test_transform: torchvision transforms for testing data.
        batch_size: Number of samples per batch in each DataLoader.
        num_workers: Number of CPU processes to use for data loading.

    Returns:
        A tuple of (train_dataloader, test_dataloader, class_names).
        Where class_names is a list of the target classes.

    Example usage:
        train_dataloader, test_dataloader, class_names = create_dataloaders(
            dataset="CIFAR10",
            data_path="./data",
            train_transform=some_train_transform,
            test_transform=some_test_transform,
            batch_size=32,
            num_workers=2
        )
    """
    # Get dataset class safely
    dataset_class = getattr(datasets, dataset_name)

    # Create datasets
    train_data = dataset_class(
        root=data_path,
        train=True,
        transform=train_transform,
        download=True
    )
    test_data = dataset_class(
        root=data_path,
        train=False,
        transform=test_transform,
        download=True
    )

    # Get class names
    class_names = train_data.classes

    # Create DataLoaders
    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,  # No need to shuffle test data
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_dataloader, test_dataloader, class_names

Overwriting modules/data_setup.py


In [50]:
# from modules import data_setup

# import importlib
# importlib.reload(data_setup)  # reloads the latest file content

# # Create train/test dataloader and get class names as a list
# train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
#         dataset_name="CIFAR10",
#         data_path="data",
#         train_transform=transforms.ToTensor(),
#         test_transform=transforms.ToTensor(),
#         batch_size=32,
#         )

# len(train_dataloader), len(test_dataloader)

In [51]:
# display_random_imgs(train_dataloader, sample_size=3)

## **2. Making a model (model_builder.py)**

In [52]:
%%writefile modules/model_builder.py
"""
Contains PyTorch model code to instantiate a TinyVGG model.
"""
import torch
from torch import nn

class TinyVGG(nn.Module):
  """Creates the TinyVGG architecture.

  Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
  See the original architecture here: https://poloclub.github.io/cnn-explainer/

  Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
      super().__init__()
      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                        stride=2)
      )
      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )
      self.classifier = nn.Sequential(
          nn.Flatten(),
          # Where did this in_features shape come from?
          # It's because each layer of our network compresses and changes the shape of our inputs data.
          nn.Linear(in_features=hidden_units*5*5,
                    out_features=output_shape)
      )

  def forward(self, x: torch.Tensor):
      # x = self.conv_block_1(x)
      # x = self.conv_block_2(x)
      # x = self.classifier(x)
      # return x
      return self.classifier(self.conv_block_2(self.conv_block_1(x))) # <- leverage the benefits of operator fusion

Overwriting modules/model_builder.py


In [53]:
# import torch
# from modules import model_builder

# import importlib
# importlib.reload(model_builder)  # reloads the latest file content

# # Instantiate an instance of the model from the "model_builder.py" script
# torch.manual_seed(42)
# model = model_builder.TinyVGG(input_shape=3,
#                               hidden_units=10,
#                               output_shape=len(class_names)).to(DEVICE)

# summary(model, input_size=(1, 3, 32, 32))

## **3. Creating train_step(), test_step() and eval() functions and train() to combine them**

In [54]:
%%writefile modules/engine.py
"""
Contains functions for training and testing a PyTorch model.
"""
import numpy as np
import torch
from typing import Dict, List, Tuple
import time

def train_step(model: torch.nn.Module,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               dataloader: torch.utils.data.DataLoader,
               device: torch.device = "cpu"
               ) -> Tuple[float, float]:
  # model.to(device)
  model.train()

  total_loss = 0.0
  total_correct_predictions = 0
  total_samples = 0

  for batch, (data, target) in enumerate(dataloader):
    data, target = data.to(device), target.to(device)

    # Forward pass
    y_logits = model(data)

    # Calculate the Loss
    loss = loss_fn(y_logits, target)
    total_loss += (loss.item() * target.size(0))

    # Count correct
    total_correct_predictions += (torch.argmax(y_logits, dim=1) == target).sum().item()
    total_samples += target.size(0)

    # Zero Grad
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Apply Grad
    optimizer.step()

  avg_loss = total_loss / total_samples
  accuracy = total_correct_predictions / total_samples * 100

  return (avg_loss, accuracy)


def test_step(model: torch.nn.Module,
              loss_fn: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              device: torch.device = "cpu"
              ) -> Tuple[float, float]:

  # model.to(device)
  model.eval()

  total_loss = 0.0
  total_correct_predictions = 0
  total_samples = 0

  with torch.inference_mode():
    for batch, (data, target) in enumerate(dataloader):
      data, target = data.to(device), target.to(device)

      # Forward pass
      y_logits = model(data)

      # Calculate the Loss
      loss = loss_fn(y_logits, target)
      total_loss += (loss.item() * target.size(0))

      # Count correct
      total_correct_predictions += (torch.argmax(y_logits, dim=1) == target).sum().item()
      total_samples += target.size(0)

  avg_loss = total_loss / total_samples
  accuracy = total_correct_predictions / total_samples * 100

  return (avg_loss, accuracy)


def eval_model(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               device: torch.device = "cpu") -> tuple[np.ndarray, torch.Tensor, torch.Tensor]:
    """
    Return probability, prediction, and target of all samples as 1D tensors.
    """
    model.to(device)
    model.eval()

    y_probs = []
    y_preds = []
    targets = []

    with torch.inference_mode():
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)

            # Forward pass
            y_logits = model(data)

            # Softmax probabilities and predictions
            probs = torch.softmax(y_logits, dim=1).max(dim=1).values
            preds = torch.argmax(y_logits, dim=1)

            y_probs.append(probs)
            y_preds.append(preds)
            targets.append(target)

    # Convert to proper types
    y_prob_np = torch.cat(y_probs).cpu().numpy()
    y_pred_tensor = torch.cat(y_preds).cpu()
    target_tensor = torch.cat(targets).cpu()

    return y_prob_np, y_pred_tensor, target_tensor


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device,
          scheduler: torch.optim.lr_scheduler=None) -> Dict[str, List]:
  """Trains and tests a PyTorch model.

  Passes a target PyTorch models through train_step() and test_step()
  functions for a number of epochs, training and testing the model
  in the same epoch loop.

  Calculates, prints and stores evaluation metrics throughout.

  Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").
    scheduler: Adjust the learning rate based on the number of epochs.

  Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for
    each epoch.
    In the form: {train_loss: [...],
                  train_acc: [...],
                  test_loss: [...],
                  test_acc: [...]}
    For example if training for epochs=2:
                 {train_loss: [2.0616, 1.0537],
                  train_acc: [0.3945, 0.3945],
                  test_loss: [1.2641, 1.5706],
                  test_acc: [0.3400, 0.2973]}
  """
  # Create empty results dictionary
  results = {"train_loss": [],
      "train_acc": [],
      "test_loss": [],
      "test_acc": [],
      "time": []
  }

  # Loop through training and testing steps for a number of epochs
  for epoch in range(epochs):
      start_time=time.time()
      train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
      if scheduler:
        scheduler.step()

      test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

      # Print out what's happening
      print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}"
      )

      delta_time = time.time() - start_time
      delta_time = round(delta_time, 2)
      print(f"Epoch time: {delta_time}s\n")

      # Update results dictionary
      results["train_loss"].append(train_loss)
      results["train_acc"].append(train_acc)
      results["test_loss"].append(test_loss)
      results["test_acc"].append(test_acc)
      results["time"].append(delta_time)

  # Return the filled results at the end of the epochs
  return results

Overwriting modules/engine.py


In [55]:
# # Import engine.py
# from modules import engine

# import importlib
# importlib.reload(engine)  # reloads the latest file content

# optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

# loss_fn = torch.nn.CrossEntropyLoss()

# # Use train() by calling it from engine.py
# results = engine.train(
#     model=model,
#     train_dataloader=train_dataloader,
#     test_dataloader=test_dataloader,
#     optimizer=optimizer,
#     loss_fn=loss_fn,
#     epochs=2,
#     device=DEVICE,
#     scheduler=None
# )

In [56]:
# from sklearn.metrics import classification_report

# probs, preds, targets = engine.eval_model(model=model, dataloader=test_dataloader, device=DEVICE)
# print(classification_report(y_pred=preds.numpy(), y_true=targets.numpy(), target_names=class_names))

## **4. Creating a function to save the model (utils.py)**

In [57]:
%%writefile modules/utils.py
"""
Contains various utility functions for PyTorch model training and saving.
"""
import torch
from pathlib import Path

import numpy as np
import random
import matplotlib.pyplot as plt

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
  """Saves a PyTorch model to a target directory.

  Args:
    model: A target PyTorch model to save.
    target_dir: A directory for saving the model to.
    model_name: A filename for the saved model. Should include
      either ".pth" or ".pt" as the file extension.

  Example usage:
    save_model(model=model_0,
               target_dir="models",
               model_name="05_going_modular_tingvgg_model.pth")
  """
  # Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.cpu().state_dict(),
             f=model_save_path)


def display_random_imgs(dataloader: torch.utils.data.DataLoader,
                        sample_size: int = 9,
                        num_of_cols: int = 3,
                        display_shape: bool=True,
                        seed: int=None):
    """Display random images from a DataLoader (by accessing its dataset)."""
    dataset = dataloader.dataset  # <-- underlying dataset
    if seed:
        random.seed(seed)
        np.random.seed(seed)

    idxs = np.random.randint(low=0, high=len(dataset), size=(sample_size,))
    cols = num_of_cols
    rows = (sample_size // cols) + 1 if sample_size % cols != 0 else sample_size // cols

    plt.figure(figsize=(3 * cols, 3 * rows))

    for i, image_idx in enumerate(idxs):
        img, label = dataset[image_idx]
        img = img.cpu().numpy()
        class_name = dataset.classes[label]

        plt.subplot(rows, cols, i + 1)
        img = np.transpose(img, (1, 2, 0))
        plt.imshow(img)
        if display_shape:
            class_name += " " + str(img.shape)
        plt.title(class_name, fontsize=10)
        plt.axis(False)

    plt.show()

Overwriting modules/utils.py


In [58]:
# # Import utils.py
# from modules import utils

# import importlib
# importlib.reload(utils)  # reloads the latest file content

# # Save a model to file
# utils.save_model(model=model,
#            target_dir="./",
#            model_name="test_model.pth")

## **5. Train, evaluate and save the model (train.py)**

Combine all of the functionality together in a train.py file + argparse

In [77]:
%%writefile modules/train.py
"""
Trains a PyTorch image classification model using device-agnostic code.
"""

import os
import torch
import argparse
import data_setup, engine, model_builder, utils
from pathlib import Path

import importlib
importlib.reload(data_setup)
importlib.reload(engine)
importlib.reload(model_builder)
importlib.reload(utils)

from torchvision import transforms

def get_args():
    parser = argparse.ArgumentParser(description="Train a PyTorch image classifier")

    # Hyperparameters
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--epochs", type=int, default=5, help="Number of epochs")
    parser.add_argument("--batch-size", type=int, default=32, help="Batch size")
    parser.add_argument("--hidden-units", type=int, default=10, help="Hidden units")
    parser.add_argument("--lr", type=float, default=0.1, help="Learning rate")

    # Data settings
    parser.add_argument("--data-root", type=str, default="data", help="Data root directory")
    parser.add_argument("--data-name", type=str, default="FashionMNIST", help="Dataset name")

    # Model saving
    parser.add_argument(
        "--output",
        type=str,
        default="models/tinyvgg.pth",
        help="Full path (directory + filename) to save the trained model"
    )

    return parser.parse_args()


def main():
    args = get_args()

    RANDOM_SEED = args.seed
    NUM_EPOCHS = args.epochs
    BATCH_SIZE = args.batch_size
    HIDDEN_UNITS = args.hidden_units
    LEARNING_RATE = args.lr
    DATA_ROOT = args.data_root
    DATA_NAME = args.data_name

    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)  # create directory if missing

    # Setup target device
    DEVICE = "mps" if torch.mps.is_available() else "cpu"
    DEVICE = "cuda" if torch.cuda.is_available() else DEVICE
    print(f"Using device: {DEVICE}")

    # Create transforms
    train_transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    test_transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Create DataLoaders with help from data_setup.py
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.mps.manual_seed(RANDOM_SEED)

    train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
        dataset_name=DATA_NAME,
        data_path=DATA_ROOT,
        train_transform=train_transform,
        test_transform=test_transform,
        batch_size=BATCH_SIZE
    )

    # Create model with help from model_builder.py
    model = model_builder.TinyVGG(
        input_shape=1,
        hidden_units=HIDDEN_UNITS,
        output_shape=len(class_names)
    ).to(DEVICE)

    # Set loss and optimizer
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

    # Start training with help from engine.py
    engine.train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=NUM_EPOCHS,
                device=DEVICE)

    # Save the model with help from utils.py
    # Save the model
    utils.save_model(
        model=model,
        target_dir=str(output_path.parent),
        model_name=output_path.name
    )


if __name__ == "__main__":
    main()

Overwriting modules/train.py


## **6. Run train.py**

In [78]:
lr = 0.1
num_epochs = 5
batch_size = 32
hidden_units = 10
random_seed = 42
dataset_name = "FashionMNIST"
data_root = "data"
output = "checkpoints/fashion_vgg.pth"

In [79]:
!python modules/train.py --epochs {num_epochs} --batch-size {batch_size} --hidden-units {hidden_units} --lr {lr} --data-name {dataset_name}

Using device: cuda
Epoch: 1 | train_loss: 0.6694 | train_acc: 75.8917 | test_loss: 0.4393 | test_acc: 84.2900
Epoch time: 19.83s

Epoch: 2 | train_loss: 0.3977 | train_acc: 85.7917 | test_loss: 0.4118 | test_acc: 85.6000
Epoch time: 20.76s

Epoch: 3 | train_loss: 0.3570 | train_acc: 87.2133 | test_loss: 0.3494 | test_acc: 87.3000
Epoch time: 23.23s

Epoch: 4 | train_loss: 0.3308 | train_acc: 88.2050 | test_loss: 0.3450 | test_acc: 87.7800
Epoch time: 24.27s

Epoch: 5 | train_loss: 0.3174 | train_acc: 88.7050 | test_loss: 0.3274 | test_acc: 88.4600
Epoch time: 20.41s

[INFO] Saving model to: models/tinyvgg.pth
